In [5]:
%pip install pymysql
%pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

connection_url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password="XXXXXXX",
    host="localhost",
    port=3306,
    database="new_schema"
)

engine = create_engine(connection_url)

df = pd.read_sql(
    "SELECT * FROM superstore_staging_0",
    engine
)

print(df.head())
print(df.shape)

   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156  2016-11-08  2016-11-11    Second Class    CG-12520   
1       2  CA-2016-152156  2016-11-08  2016-11-11    Second Class    CG-12520   
2       3  CA-2016-138688  2016-06-12  2016-06-16    Second Class    DV-13045   
3       4  US-2015-108966  2015-10-11  2015-10-18  Standard Class    SO-20335   
4       5  US-2015-108966  2015-10-11  2015-10-18  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  Postal Code  Region       Product ID         Category Sub-Category  \
0       42420   Sout

In [11]:
df = df.drop(columns=['Row ID','Order ID','Customer ID','Customer Name','City','Product Name','Product ID','Country'])


In [12]:

high_threshold = df.loc[df["Profit"] > 0, "Profit"].median()

print(high_threshold)

13.452


In [13]:
def profit_class(profit):
    if profit < 0:
        return 0
    elif profit < high_threshold:
        return 1
    else:
        return 2

df["Profit Class"] = df["Profit"].apply(profit_class)

In [20]:
df = df.drop(columns=['Postal Code'])

In [17]:
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

# Month when customer ordered
df["Order Month"] = df["Order Date"].dt.month

# Number of days between order and shipping
df["Delivery Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

# Remove original dates
df.drop(columns=["Order Date", "Ship Date"], inplace=True)

In [29]:

label = df['Profit Class']
df

,Ship Mode,Segment,State,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit Class,Order Month,Delivery Days
0,Second Class,Consumer,Kentucky,South,Furniture,Bookcases,261.9600,2,0.00,2,11,3
1,Second Class,Consumer,Kentucky,South,Furniture,Chairs,731.9400,3,0.00,2,11,3
2,Second Class,Corporate,California,West,Office Supplies,Labels,14.6200,2,0.00,1,6,4
3,Standard Class,Consumer,Florida,South,Furniture,Tables,957.5775,5,0.45,0,10,7
4,Standard Class,Consumer,Florida,South,Office Supplies,Storage,22.3680,2,0.20,1,10,7
...,...,...,...,...,...,...,...,...,...,...,...,...
19383,Second Class,Consumer,Florida,South,Furniture,Furnishings,25.2480,3,0.20,1,1,2
19384,Standard Class,Consumer,California,West,Furniture,Furnishings,91.9600,2,0.00,2,2,5
19385,Standard Class,Consumer,California,West,Technology,Phones,258.5760,2,0.20,2,2,5
19386,Standard Class,Consumer,California,West,Office Supplies,Paper,29.6000,4,0.00,1,2,5


In [32]:
df.drop(columns='Profit Class', inplace=True)


In [35]:
categorical_cols = [
    "Ship Mode",
    "Segment",
    "State",
    "Region",
    "Category",
    "Sub-Category",
    "Order Month"
]

df_0 = pd.get_dummies(
    df,
    columns=categorical_cols,
    dtype=int
)

In [38]:
import torch 

In [39]:
X_tensor = torch.tensor(df_0.values, dtype=torch.float32)
y_tensor = torch.tensor(label.values, dtype=torch.long)


In [43]:
X_tensor.size()

torch.Size([19388, 96])

In [44]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_tensor,
    y_tensor,
    test_size=0.2,
    random_state=42,
    stratify=y_tensor
)

In [50]:
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(96, 64),
    nn.ReLU(),

    nn.Linear(64, 32),
    nn.ReLU(),

    nn.Linear(32, 3)
)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 10000

for epoch in range(epochs):

    # Forward
    outputs = model(X_train)

    # Loss
    loss = loss_fn(outputs, y_train)

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

Epoch 10/10000 | Loss: 1.0059
Epoch 20/10000 | Loss: 0.9547
Epoch 30/10000 | Loss: 0.8486
Epoch 40/10000 | Loss: 0.7989
Epoch 50/10000 | Loss: 0.7574
Epoch 60/10000 | Loss: 0.7120
Epoch 70/10000 | Loss: 0.6647
Epoch 80/10000 | Loss: 0.6208
Epoch 90/10000 | Loss: 0.5820
Epoch 100/10000 | Loss: 0.5514
Epoch 110/10000 | Loss: 0.5461
Epoch 120/10000 | Loss: 0.4896
Epoch 130/10000 | Loss: 0.4670
Epoch 140/10000 | Loss: 0.4555
Epoch 150/10000 | Loss: 0.4148
Epoch 160/10000 | Loss: 0.3987
Epoch 170/10000 | Loss: 0.3839
Epoch 180/10000 | Loss: 0.3599
Epoch 190/10000 | Loss: 0.3489
Epoch 200/10000 | Loss: 0.5099
Epoch 210/10000 | Loss: 0.3465
Epoch 220/10000 | Loss: 0.3326
Epoch 230/10000 | Loss: 0.3200
Epoch 240/10000 | Loss: 0.3124
Epoch 250/10000 | Loss: 0.3059
Epoch 260/10000 | Loss: 0.3002
Epoch 270/10000 | Loss: 0.2952
Epoch 280/10000 | Loss: 0.2909
Epoch 290/10000 | Loss: 0.2869
Epoch 300/10000 | Loss: 0.2833
Epoch 310/10000 | Loss: 0.2799
Epoch 320/10000 | Loss: 0.2767
Epoch 330/10000 |

In [51]:
with torch.no_grad():

    outputs = model(X_test)

    predictions = outputs.argmax(dim=1)

    accuracy = (predictions == y_test).float().mean()

    print("Test Accuracy:", accuracy.item())

Test Accuracy: 0.9463641047477722
